# Volume 1, Chapter 12: The p-adic Numbers (Python companion)

Exercises first, solutions at the end. Computer-science threads: **fixed-width two's complement represents truncated 2-adic integers**,
**Newton's method in $\Z_p$** (quadratic convergence, digit by digit), $\Z_p$ **drawn as a fractal**, and **Dixon's
$p$-adic algorithm**, which solves integer linear systems exactly by lifting a solution modulo $p$ to modulo $p^k$.

*Before each solution, predict what the output will look like.*

In [ ]:
import math
import random
import time
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt
# Reproducible teaching examples; these seeds are not for cryptographic keys.
import random
from sympy.core.random import seed as seed_sympy
random.seed(10112)
seed_sympy(20112)


## Before the experiment: Nearness has changed its mind

Predict which is closer to 1 in the 2-adic metric: 9 or 10. Decimal size is no longer a useful guide. Fixed-width arithmetic models a finite quotient $\mathbb Z/2^k\mathbb Z$, not the whole infinite ring $\mathbb Z_2$.

**How to use this notebook.** Try the exercise cells first; they are intentionally blank. Then run the solutions in order in a fresh kernel. Assertions check exact identities or a stated finite range; plots and random trials supply evidence, not proofs.


## Exercises

**P12.1 (★) $p$-adic digits.** Write `digits(x, p, n)` returning the first $n$ digits of a rational $x$ with $p\nmid$ denominator. Check $-1$, $\frac13$ in $\Z_2$ and $\frac14$ in $\Z_5$ (Exercise 12.1), then find the eventual period of $\frac1{q}$ in $\Z_2$ for odd $q\le 31$. *Guess: what is the period, in terms of $q$?* Finally, look at how Python stores $-1$, $-3$ and $\frac13$ modulo $2^{64}$.

In [ ]:
# Your code here


**P12.2 (★) Newton in $\Z_2$.** Run Newton's iteration for $\sqrt{-7}$ from $a_0=1$ (Theorem 12.7) with exact fractions, and print the number of correct $2$-adic digits after each step. Check that it roughly doubles.

In [ ]:
# Your code here


**P12.3 (★★) Pictures of $\Z_p$.** (a) Draw $\Z_3$ in the plane by sending $\sum a_i3^i$ to $\sum a_i\,r^i e^{2i\pi a_i/3}$ with $r=0.42$, for all $3^8$ residues modulo $3^8$. (b) Plot the *Monna map* $\sum a_i2^i\mapsto\sum a_i2^{-i-1}$ for $0\le n<2^{10}$: it is continuous from $\Z_2$ to $[0,1]$. What does the picture say about which integers are $2$-adically close?

In [ ]:
# Your code here


**P12.4 (★) Teichmüller representatives.** For $p=7$, compute $\omega(a)=\lim a^{7^n}$ modulo $7^{12}$ for $a=1,\dots,6$ (Exercise 12.7) and check $\omega(a)^6=1$ and $\omega(a)\equiv a\pmod 7$. Which $\omega(a)$ is a cube root of unity?

In [ ]:
# Your code here


**P12.5 (★★★) CS application: Dixon's $p$-adic linear solver.** To solve $Ax=b$ exactly for an integer matrix $A$, Dixon (1982) inverts $A$ only modulo a small prime $p$, then lifts: if $Ax\equiv b\pmod{p^i}$, one more digit is found modulo $p$. After $k$ steps, *rational reconstruction* (the extended Euclidean algorithm of Chapter 1, stopped halfway) turns residues modulo $p^k$ into fractions. Implement it, and compare its speed with Gaussian elimination over $\Q$ for random $40\times40$ matrices with entries in $[-99,99]$.

*How to think about it:* Gaussian elimination over $\Q$ suffers from growing fractions; Dixon's method only ever handles numbers modulo $p$ plus one matrix-vector product per digit. This is Hensel's lemma for the linear polynomial $Ax-b$.

In [ ]:
# Your code here


---
## Solutions

### P12.1
The first digit is $x\bmod p$, then we replace $x$ by $\frac{x-a_0}{p}$ (proof of Proposition 12.6).

In [ ]:
def digits(x, p, n):
    x = Fraction(x)
    out = []
    for _ in range(n):
        a = x.numerator * pow(x.denominator, -1, p) % p
        out.append(a)
        x = (x - a) / p
    return out

print("-1  in Z_2:", digits(-1, 2, 16))
print("1/3 in Z_2:", digits(Fraction(1, 3), 2, 16))
print("1/4 in Z_5:", digits(Fraction(1, 4), 5, 12))

def eventual_period(x, p, n=400):
    d = digits(x, p, n)
    tail = d[n // 2:]
    for ell in range(1, n // 4):
        if all(tail[i] == tail[i + ell] for i in range(len(tail) - ell)):
            return ell

for q in range(3, 32, 2):
    order = next(k for k in range(1, q) if pow(2, k, q) == 1)
    print(f"q = {q:2d}: period {eventual_period(Fraction(1, q), 2):2d}, order of 2 mod q = {order}")

M = 2**64
print("two's complement of -1:", format(-1 % M, '064b'))
print("two's complement of -3:", format(-3 % M, '064b'))
print("1/3 mod 2^64          :", format(pow(3, -1, M), '064b'))

The period of $\frac1q$ is the multiplicative order of $2$ modulo $q$: indeed $\frac1q=\frac{c}{1-2^\ell}$ exactly when $q\mid2^\ell-1$. And a fixed-width representation of $-1$ is the $2$-adic $-1$ truncated to $64$ digits: unsigned arithmetic modulo $2^{64}$ is arithmetic in $\Z_2$ to precision $64$. The inverse of $3$ modulo $2^{64}$ is `0xAAAAAAAAAAAAAAAB`. This is a modular inverse, not ordinary integer division by $3$. Compilers can combine related multiplication constants with shifts and high-word extraction to implement integer division. Python integers themselves have arbitrary precision; the `% 2**64` operation here explicitly creates the fixed-width model.

### P12.2

In [ ]:
def v2(x):
    x = Fraction(x)
    if x == 0:
        return math.inf
    v, n, d = 0, x.numerator, x.denominator
    while n % 2 == 0:
        n //= 2; v += 1
    while d % 2 == 0:
        d //= 2; v -= 1
    return v

a = Fraction(1)
for j in range(7):
    # a is correct to about v2(a^2 + 7) - 1 digits (|f(a)| = |2a| |a - alpha|)
    print(f"step {j}: correct 2-adic digits of sqrt(-7) = {v2(a * a + 7) - 1}")
    a = (a * a - 7) / (2 * a)

The digit counts go $2,\ 3,\ 5,\ 9,\ 17,\ 33,\ 65$: each step doubles the number of correct digits minus one, the $p$-adic face of Newton's quadratic convergence (Theorem 12.7's estimate $|f(a_j)|\le\delta^2t^{2^j}$).

### P12.3

In [ ]:
p, n, r = 3, 8, 0.42
pts = []
for m in range(p**n):
    z, ds, x = 0, [], m
    for i in range(n):
        a = x % p; x //= p
        z += a * r**i * np.exp(2j * np.pi * a / p)
    pts.append(z)
pts = np.array(pts)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].scatter(pts.real, pts.imag, s=0.3, c=np.arange(p**n) % 9, cmap='viridis')
ax[0].set_aspect('equal'); ax[0].axis('off'); ax[0].set_title(r'$\mathbb{Z}_3$: each ball splits into 3 smaller balls')

def monna(m, bits=10):
    return sum(((m >> i) & 1) * 2.0**(-i - 1) for i in range(bits))
ns = np.arange(2**10)
ax[1].plot(ns, [monna(int(m)) for m in ns], '.', ms=1.5)
ax[1].set_xlabel('n'); ax[1].set_ylabel('Monna map'); ax[1].set_title(r'$\mathbb{Z}_2\to[0,1]$, reversing the binary digits')
plt.tight_layout(); plt.show()

The first picture is a fractal because $\Z_3$ is self-similar: $\Z_3=\bigsqcup_{a}(a+3\Z_3)$, three copies of itself scaled by $|3|_3=\frac13$ (compare Figure 12.1). In the second, integers with the same last digits (which are $2$-adically close) land close together: $n$ and $n+2^k$ are mapped within $2^{-k}$ of each other. The Monna map is continuous and onto, but not injective (dyadic rationals have two expansions).

### P12.4

In [ ]:
p, prec = 7, 12
mod = p**prec
for a in range(1, p):
    w = a
    for _ in range(prec):
        w = pow(w, p, mod)
    assert pow(w, p - 1, mod) == 1 and w % p == a
    cube = w != 1 and pow(w, 3, mod) == 1
    print(f"omega({a}) = {w:>12d} mod 7^12   order {next(k for k in (1,2,3,6) if pow(w, k, mod) == 1)}"
          + ("   (a primitive cube root of unity)" if cube else ""))

$\omega(2)$ and $\omega(4)$ are the primitive cube roots of unity in $\Z_7$, because $2$ and $4$ have order $3$ modulo $7$: so $\Q_7$ contains a root of $X^2+X+1$, as the remark after Corollary 12.9 said.

### P12.5

In [ ]:
def inv_mod_matrix(A, p):
    from sympy import isprime
    n = len(A)
    if not isprime(p) or n == 0 or any(len(row) != n for row in A):
        raise ValueError("require a nonempty square matrix and a prime modulus")
    M = [[x % p for x in row] + [int(i == j) for j in range(n)] for i, row in enumerate(A)]
    for c in range(n):
        piv = next((r for r in range(c, n) if M[r][c]), None)
        if piv is None:
            raise ValueError("matrix is singular modulo p; choose another prime")
        M[c], M[piv] = M[piv], M[c]
        inv = pow(M[c][c], -1, p)
        M[c] = [x * inv % p for x in M[c]]
        for r in range(n):
            if r != c and M[r][c]:
                f = M[r][c]
                M[r] = [(x - f * y) % p for x, y in zip(M[r], M[c])]
    return np.array([row[n:] for row in M], dtype=object)

def rational_reconstruction(u, m):
    # find a/b with a = b*u mod m, |a|, b <= sqrt(m/2): extended Euclid stopped halfway
    if m < 3:
        raise ValueError("reconstruction modulus must be at least 3")
    bound = math.isqrt((m - 1) // 2)  # ensure 2*bound^2 < m for uniqueness
    r0, r1, t0, t1 = m, u % m, 0, 1
    while r1 > bound:
        q = r0 // r1
        r0, r1, t0, t1 = r1, r0 - q * r1, t1, t0 - q * t1
    if t1 == 0 or abs(t1) > bound or math.gcd(r1, t1) != 1:
        return None
    candidate = Fraction(r1, t1)
    if math.gcd(candidate.denominator, m) != 1:
        return None
    if (candidate.numerator - candidate.denominator*u) % m != 0:
        return None
    return candidate

def dixon(A, b, p=1000003):
    A = np.array(A, dtype=object); b = np.array(b, dtype=object)
    C = inv_mod_matrix(A.tolist(), p)
    x, res, pk, k = np.zeros(len(b), dtype=object), b.copy(), 1, 0
    while True:
        for _ in range(8):                          # eight more p-adic digits
            xi = C.dot(res) % p
            x = x + xi * pk
            res = (res - A.dot(xi)) // p             # exact division: A xi = res mod p
            pk *= p; k += 1
        sol = [rational_reconstruction(int(u), pk) for u in x]
        if None not in sol and all(sum(Fraction(int(a)) * s for a, s in zip(row, sol)) == int(bi)
                                   for row, bi in zip(A, b)):
            return sol, k

def gauss_Q(A, b):
    n = len(A)
    M = [[Fraction(x) for x in row] + [Fraction(bi)] for row, bi in zip(A, b)]
    for c in range(n):
        piv = next(r for r in range(c, n) if M[r][c] != 0)
        M[c], M[piv] = M[piv], M[c]
        for r in range(c + 1, n):
            f = M[r][c] / M[c][c]
            if f:
                M[r] = [x - f * y for x, y in zip(M[r], M[c])]
    x = [Fraction(0)] * n
    for i in reversed(range(n)):
        x[i] = (M[i][n] - sum(M[i][j] * x[j] for j in range(i + 1, n))) / M[i][i]
    return x

random.seed(2)
n = 40
A = [[random.randint(-99, 99) for _ in range(n)] for _ in range(n)]
b = [random.randint(-99, 99) for _ in range(n)]
t0 = time.time(); sol, k = dixon(A, b); t1 = time.time(); ref = gauss_Q(A, b); t2 = time.time()
assert sol == ref
print(f"Dixon: {t1 - t0:.2f} s ({k} p-adic digits),  Gaussian elimination over Q: {t2 - t1:.2f} s")
print("denominator of the solution has", len(str(ref[0].denominator)), "digits")

Both methods return the same solution, and we verify $Ax=b$ exactly before accepting a reconstructed vector. Each denominator divides $\det A$ (Cramer's rule), so coefficient size determines how much lifting precision is needed. The modulus must be a prime for which $A$ is invertible; a failed modular pivot means choosing another prime. Dixon's setup costs roughly $n^3$ field operations, followed by $n^2$ operations per digit, but integer sizes, reconstruction, and the matrix all affect the actual timing. One small benchmark does not establish a universal speed winner.


## One more question before closing

In rational reconstruction, why must a candidate denominator be invertible modulo the lifting modulus? Why is verifying Ax=b the final authority?

Write one sentence identifying what was proved in the chapter, what was checked exactly by the code, and what remains a conjecture or a heuristic.
